In [ ]:
import requests
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
import pandas as pd

In [ ]:
import requests
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, ArrowStyle
import xml.etree.ElementTree as ET
from collections import defaultdict
import numpy as np

In [ ]:
# KEGG pathway 데이터 가져오기
def get_kegg_pathway(pathway_id):
    """KEGG pathway의 gene relations 가져오기"""
    url = f"http://rest.kegg.jp/get/{pathway_id}/kgml"
    response = requests.get(url)
    return response.text

# 통합 pathway 그래프 생성
pathways = {
    'mTOR': 'hsa04150',
    'Autophagy': 'hsa04140',
    'Glycolysis': 'hsa00010',
    'BCAA': 'hsa00280'
}

In [ ]:
pathways

In [ ]:
class PathwayIntegrator:
    def __init__(self):
        self.G = nx.DiGraph()
        self.gene_names = {}
        self.edge_types = {}
        
    def parse_kegg_kgml(self, kgml_text):
        """KEGG KGML 파싱"""
        root = ET.fromstring(kgml_text)
        
        # Entry (gene/compound) 정보 추출
        entries = {}
        for entry in root.findall('entry'):
            entry_id = entry.get('id')
            entry_type = entry.get('type')
            
            if entry_type == 'gene':
                graphics = entry.find('graphics')
                if graphics is not None:
                    gene_name = graphics.get('name')
                    entries[entry_id] = gene_name
                    self.gene_names[gene_name] = gene_name
        
        # Relation (edge) 정보 추출
        for relation in root.findall('relation'):
            entry1 = relation.get('entry1')
            entry2 = relation.get('entry2')
            rel_type = relation.get('type')
            
            if entry1 in entries and entry2 in entries:
                gene1 = entries[entry1]
                gene2 = entries[entry2]
                
                # Subtype으로 activation/inhibition 구분
                edge_type = 'activation'
                for subtype in relation.findall('subtype'):
                    subtype_name = subtype.get('name')
                    if subtype_name == 'inhibition':
                        edge_type = 'inhibition'
                    elif subtype_name == 'activation':
                        edge_type = 'activation'
                    elif subtype_name == 'phosphorylation':
                        edge_type = 'activation'
                
                self.G.add_edge(gene1, gene2)
                self.edge_types[(gene1, gene2)] = edge_type
    
    def get_kegg_pathway(self, pathway_id):
        """KEGG pathway KGML 다운로드"""
        url = f"http://rest.kegg.jp/get/{pathway_id}/kgml"
        response = requests.get(url)
        if response.status_code == 200:
            return response.text
        return None
    
    def add_manual_connections(self):
        """pathway 간 연결 수동 추가"""
        # SLC7A5 -> mTOR pathway 연결
        connections = [
            ('SLC7A5', 'mTOR', 'activation'),
            ('mTOR', 'ULK1', 'inhibition'),  # autophagy 억제
            ('mTOR', 'HIF1A', 'activation'),  # glycolysis 활성화
            ('mTOR', 'BCKDHA', 'activation'), # BCAA metabolism
        ]
        
        for gene1, gene2, edge_type in connections:
            self.G.add_edge(gene1, gene2)
            self.edge_types[(gene1, gene2)] = edge_type
    
    def load_pathways(self, pathway_dict):
        """여러 pathway 로드"""
        for name, pathway_id in pathway_dict.items():
            print(f"Loading {name} ({pathway_id})...")
            kgml = self.get_kegg_pathway(pathway_id)
            if kgml:
                self.parse_kegg_kgml(kgml)
                print(f"  Added {self.G.number_of_nodes()} nodes")
    
    def visualize(self, figsize=(20, 16), seed=42):
        """그래프 시각화"""
        plt.figure(figsize=figsize)
        
        # Layout 계산
        pos = nx.spring_layout(self.G, k=2, iterations=50, seed=seed)
        
        # 노드 그리기
        nx.draw_networkx_nodes(self.G, pos, 
                              node_color='lightblue',
                              node_size=1500,
                              alpha=0.9)
        
        # 노드 라벨
        nx.draw_networkx_labels(self.G, pos, 
                               font_size=8,
                               font_weight='bold')
        
        # Edge 타입별로 그리기
        activation_edges = [(u, v) for u, v in self.G.edges() 
                           if self.edge_types.get((u, v)) == 'activation']
        inhibition_edges = [(u, v) for u, v in self.G.edges() 
                           if self.edge_types.get((u, v)) == 'inhibition']
        
        # Activation edges (일반 화살표)
        nx.draw_networkx_edges(self.G, pos,
                              edgelist=activation_edges,
                              edge_color='green',
                              arrows=True,
                              arrowsize=20,
                              arrowstyle='->',
                              width=2,
                              alpha=0.6)
        
        # Inhibition edges (T자 화살표)
        nx.draw_networkx_edges(self.G, pos,
                              edgelist=inhibition_edges,
                              edge_color='red',
                              arrows=True,
                              arrowsize=20,
                              arrowstyle='|-|',
                              width=2,
                              alpha=0.6)
        
        plt.title("Integrated Pathway: SLC7A5 → mTOR → Metabolism", 
                 fontsize=16, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        
        return plt

In [ ]:
# 실행 코드
if __name__ == "__main__":
    # Pathway 통합기 생성
    integrator = PathwayIntegrator()
    
    # 관심 pathway들
    pathways = {
        'mTOR signaling': 'hsa04150',
        'Autophagy': 'hsa04140',
        'Glycolysis': 'hsa00010',
        'BCAA degradation': 'hsa00280',
    }
    
    # Pathway 로드
    integrator.load_pathways(pathways)
    
    # 수동 연결 추가 (pathway 간 연결)
    integrator.add_manual_connections()
    
    print(f"\nTotal nodes: {integrator.G.number_of_nodes()}")
    print(f"Total edges: {integrator.G.number_of_edges()}")
    
    # 시각화
    plt = integrator.visualize()
    plt.savefig('integrated_pathway.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# 실행 코드
if __name__ == "__main__":
    # Pathway 통합기 생성
    integrator = PathwayIntegrator()
    
    # 관심 pathway들
    pathways = {
        'mTOR signaling': 'hsa04150',
        'Autophagy': 'hsa04140'
    }
    
    # Pathway 로드
    integrator.load_pathways(pathways)
    
    # 수동 연결 추가 (pathway 간 연결)
    integrator.add_manual_connections()
    
    print(f"\nTotal nodes: {integrator.G.number_of_nodes()}")
    print(f"Total edges: {integrator.G.number_of_edges()}")
    
    # 시각화
    plt = integrator.visualize()
    plt.savefig('integrated_pathway.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
import requests
import networkx as nx
import xml.etree.ElementTree as ET
import pandas as pd
from collections import defaultdict

class PathwayIntegrator:
    def __init__(self):
        self.G = nx.DiGraph()
        self.gene_names = {}
        self.edge_types = {}
        self.node_pathways = defaultdict(set)  # 각 노드가 속한 pathway 추적
        
    def parse_kegg_kgml(self, kgml_text, pathway_name):
        """KEGG KGML 파싱"""
        root = ET.fromstring(kgml_text)
        
        # Entry (gene/compound) 정보 추출
        entries = {}
        for entry in root.findall('entry'):
            entry_id = entry.get('id')
            entry_type = entry.get('type')
            
            if entry_type == 'gene':
                graphics = entry.find('graphics')
                if graphics is not None:
                    gene_name = graphics.get('name')
                    # KEGG는 여러 gene을 쉼표로 구분할 수 있음
                    genes = gene_name.split(', ')
                    for gene in genes:
                        entries[entry_id] = gene
                        self.gene_names[gene] = gene
                        self.node_pathways[gene].add(pathway_name)
        
        # Relation (edge) 정보 추출
        for relation in root.findall('relation'):
            entry1 = relation.get('entry1')
            entry2 = relation.get('entry2')
            rel_type = relation.get('type')
            
            if entry1 in entries and entry2 in entries:
                gene1 = entries[entry1]
                gene2 = entries[entry2]
                
                # Subtype으로 activation/inhibition 구분
                edge_type = 'activation'
                interaction_detail = []
                
                for subtype in relation.findall('subtype'):
                    subtype_name = subtype.get('name')
                    interaction_detail.append(subtype_name)
                    
                    if subtype_name == 'inhibition':
                        edge_type = 'inhibition'
                    elif subtype_name == 'activation':
                        edge_type = 'activation'
                    elif subtype_name == 'phosphorylation':
                        edge_type = 'phosphorylation'
                    elif subtype_name == 'dephosphorylation':
                        edge_type = 'dephosphorylation'
                    elif subtype_name == 'indirect effect':
                        edge_type = 'indirect'
                    elif subtype_name == 'binding/association':
                        edge_type = 'binding'
                
                self.G.add_edge(gene1, gene2)
                self.edge_types[(gene1, gene2)] = {
                    'type': edge_type,
                    'detail': ';'.join(interaction_detail),
                    'pathway': pathway_name
                }
    
    def get_kegg_pathway(self, pathway_id):
        """KEGG pathway KGML 다운로드"""
        url = f"http://rest.kegg.jp/get/{pathway_id}/kgml"
        response = requests.get(url)
        if response.status_code == 200:
            return response.text
        return None
    
    def add_manual_connections(self):
        """pathway 간 연결 수동 추가"""
        connections = [
            ('SLC7A5', 'mTOR', 'activation', 'amino acid transport'),
            ('SLC7A5', 'MTOR', 'activation', 'amino acid transport'),
            ('mTOR', 'ULK1', 'inhibition', 'autophagy regulation'),
            ('MTOR', 'ULK1', 'inhibition', 'autophagy regulation'),
            ('mTOR', 'HIF1A', 'activation', 'glycolysis regulation'),
            ('MTOR', 'HIF1A', 'activation', 'glycolysis regulation'),
            ('mTOR', 'BCKDHA', 'activation', 'BCAA metabolism'),
            ('MTOR', 'BCKDHA', 'activation', 'BCAA metabolism'),
            ('mTOR', 'RPS6KB1', 'activation', 'mTORC1 signaling'),
            ('MTOR', 'RPS6KB1', 'activation', 'mTORC1 signaling'),
        ]
        
        for gene1, gene2, edge_type, detail in connections:
            if not self.G.has_edge(gene1, gene2):
                self.G.add_edge(gene1, gene2)
                self.edge_types[(gene1, gene2)] = {
                    'type': edge_type,
                    'detail': detail,
                    'pathway': 'manual_connection'
                }
                self.node_pathways[gene1].add('SLC7A5_mTOR_axis')
                self.node_pathways[gene2].add('SLC7A5_mTOR_axis')
    
    def load_pathways(self, pathway_dict):
        """여러 pathway 로드"""
        for name, pathway_id in pathway_dict.items():
            print(f"Loading {name} ({pathway_id})...")
            kgml = self.get_kegg_pathway(pathway_id)
            if kgml:
                self.parse_kegg_kgml(kgml, name)
                print(f"  Current total nodes: {self.G.number_of_nodes()}")
                print(f"  Current total edges: {self.G.number_of_edges()}")
    
    def export_to_excel(self, filename='pathway_network.xlsx'):
        """Cytoscape용 엑셀 파일 생성"""
        
        # 1. Node 테이블
        node_data = []
        for node in self.G.nodes():
            node_data.append({
                'node_id': node,
                'node_name': node,
                'pathways': ';'.join(sorted(self.node_pathways[node])),
                'num_pathways': len(self.node_pathways[node]),
                'degree': self.G.degree(node),
                'in_degree': self.G.in_degree(node),
                'out_degree': self.G.out_degree(node)
            })
        
        df_nodes = pd.DataFrame(node_data)
        
        # 2. Edge 테이블
        edge_data = []
        for source, target in self.G.edges():
            edge_info = self.edge_types.get((source, target), {})
            edge_data.append({
                'source': source,
                'target': target,
                'interaction_type': edge_info.get('type', 'unknown'),
                'interaction_detail': edge_info.get('detail', ''),
                'pathway': edge_info.get('pathway', ''),
                'edge_id': f"{source}_{target}"
            })
        
        df_edges = pd.DataFrame(edge_data)
        
        # 3. 통계 정보
        stats_data = {
            'Metric': [
                'Total Nodes',
                'Total Edges',
                'Activation Edges',
                'Inhibition Edges',
                'Phosphorylation Edges',
                'Other Edges',
                'Average Degree',
                'Density'
            ],
            'Value': [
                self.G.number_of_nodes(),
                self.G.number_of_edges(),
                len([e for e in edge_data if e['interaction_type'] == 'activation']),
                len([e for e in edge_data if e['interaction_type'] == 'inhibition']),
                len([e for e in edge_data if e['interaction_type'] == 'phosphorylation']),
                len([e for e in edge_data if e['interaction_type'] not in ['activation', 'inhibition', 'phosphorylation']]),
                sum(dict(self.G.degree()).values()) / self.G.number_of_nodes() if self.G.number_of_nodes() > 0 else 0,
                nx.density(self.G)
            ]
        }
        df_stats = pd.DataFrame(stats_data)
        
        # 엑셀 파일로 저장
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            df_nodes.to_excel(writer, sheet_name='Nodes', index=False)
            df_edges.to_excel(writer, sheet_name='Edges', index=False)
            df_stats.to_excel(writer, sheet_name='Statistics', index=False)
        
        print(f"\n엑셀 파일 저장 완료: {filename}")
        print(f"- Nodes: {len(df_nodes)} rows")
        print(f"- Edges: {len(df_edges)} rows")
        
        return df_nodes, df_edges, df_stats

# 실행 코드
if __name__ == "__main__":
    # Pathway 통합기 생성
    integrator = PathwayIntegrator()
    
    # 관심 pathway들
    pathways = {
        'mTOR signaling': 'hsa04150',
        'Autophagy': 'hsa04140',
        'Glycolysis': 'hsa00010',
        'BCAA degradation': 'hsa00280',
    }
    
    # Pathway 로드
    integrator.load_pathways(pathways)
    
    # 수동 연결 추가 (pathway 간 연결)
    integrator.add_manual_connections()
    
    print(f"\n=== Final Network Statistics ===")
    print(f"Total nodes: {integrator.G.number_of_nodes()}")
    print(f"Total edges: {integrator.G.number_of_edges()}")
    
    # 엑셀로 export
    df_nodes, df_edges, df_stats = integrator.export_to_excel('SLC7A5_mTOR_integrated_pathway.xlsx')
    
    # 미리보기
    print("\n=== Node Table Preview ===")
    print(df_nodes.head(10))
    
    print("\n=== Edge Table Preview ===")
    print(df_edges.head(10))
    
    print("\n=== Statistics ===")
    print(df_stats)

In [ ]:
connections